In [13]:
import requests,os,json
import pandas as pd

In [14]:
BASE_URL = "https://ghoapi.azureedge.net/api/"
ENDPOINTS={
    "NCD_BMI_30C":    ("obesity",      "Adult"),
    "NCD_BMI_PLUS2C": ("obesity",      "Child/Adolescent"),
    "NCD_BMI_18C":    ("malnutrition", "Adult"),
    "NCD_BMI_MINUS2C":("malnutrition", "Child/Adolescent"),
}

## Fetching WHO API data with local caching 

In [15]:
def fetch_who(code, cache_dir="../data/raw"):
    os.makedirs(cache_dir,exist_ok=True)
    cache_path =f"{cache_dir}/{code}.json"
    if os.path.exists(cache_path):
        print(f" [cache hit] {code}")
        with open(cache_path) as f:
            return json.load(f)["value"]
    print(f" [fetching] {code} ...")
    resp = requests.get(BASE_URL + code, timeout=30)
    resp.raise_for_status()
    data = resp.json()  
    with open(cache_path, "w") as f:
        json.dump(data,f)
    return data["value"]   

In [16]:
frames = {}
for code, (category, age_group) in ENDPOINTS.items():
    records = fetch_who(code)
    df = pd.DataFrame(records)
    df["age_group"] = age_group
    df["_category"] = category
    frames[code] = df
    print(f"    {code}: {len(df):,} rows")

 [cache hit] NCD_BMI_30C
    NCD_BMI_30C: 20,790 rows
 [cache hit] NCD_BMI_PLUS2C
    NCD_BMI_PLUS2C: 62,370 rows
 [cache hit] NCD_BMI_18C
    NCD_BMI_18C: 20,790 rows
 [cache hit] NCD_BMI_MINUS2C
    NCD_BMI_MINUS2C: 62,370 rows


## Combining into 2 DataFrames

In [17]:
df_obesity=pd.concat([frames["NCD_BMI_30C"],frames["NCD_BMI_PLUS2C"]], ignore_index=True)
df_malnutrition=pd.concat([frames["NCD_BMI_18C"],frames["NCD_BMI_MINUS2C"]], ignore_index=True)

## Filtering from 2012-2022

In [19]:
df_obesity = df_obesity[df_obesity["TimeDim"].between(2012,2022)]
df_malnutrition = df_malnutrition[df_malnutrition["TimeDim"].between(2012,2022)]
print(f"df_obesity -> {len(df_obesity):,} rows after year filter")
print(f"df_malnutrition -> {len(df_malnutrition):,} rows after year filter")

print("Collection complete!")

df_obesity -> 27,720 rows after year filter
df_malnutrition -> 27,720 rows after year filter
Collection complete!
